In [1]:
# פרויקט: big-data-env
# מחברת: notebooks/UseCaseLLMInsights.ipynb 

from pathlib import Path
import json
from openai import OpenAI

# תיקיית ה-JSON כפי שהגדרנו בדוקר-קומפוז
OUTPUT_DIR = Path("/workspace/output/json")

# טעינת קבצי העזר
cluster_summary = json.loads((OUTPUT_DIR / "cluster_summary.json").read_text())
cluster_sizes = json.loads((OUTPUT_DIR / "cluster_sizes.json").read_text())
salary_decile_sector_summary = json.loads(
    (OUTPUT_DIR / "salary_decile_sector_summary.json").read_text()
)

# בדיקה קטנה:
len(cluster_summary), len(cluster_sizes), len(salary_decile_sector_summary)

(4, 4, 10)

In [2]:
from openai import OpenAI
client = OpenAI()

def ask_econ_assistant(question: str) -> str:
    context = build_context_text()

    system_msg = (
        "You are a senior economist and data scientist specializing in household economics. "
        "You received structured summaries from a Spark + ML pipeline that performed K-Means clustering "
        "and decile segmentation of household economic data. "
        "Use ONLY the provided data to answer. Do not hallucinate or infer beyond the input.\n\n"
        "You answer in fluent Hebrew, with intuitive and clear economic analysis, suitable for policy audiences."
    )

    user_msg = (
        "📘 הסבר מקדים:\n"
        "הקלאסטרים מייצגים קבוצות של משקי בית עם מאפיינים כלכליים דומים. "
        "לכל קלאסטר מופיעים נתונים כגון שכר ממוצע, חיסכון ממוצע, יחס הוצאה/הכנסה, שנות השכלה ודירוג כלכלי.\n"
        "העשירונים מציגים פילוח לפי הכנסה, והסקטורים מייצגים תחום תעסוקתי מוביל בכל עשירון.\n\n"

        "📄 נתוני הרקע:\n"
        f"{context}\n\n"

        "❓ שאלה:\n"
        f"{question}\n\n"

        "🧠 הנחיה לפענוח:\n"
        "שים לב: מספרי הקלאסטרים (0, 1, 2, 3) אינם מסמנים סדר או ערך. "
        "בחר שמות אינטואיטיביים וברורים לכל קלאסטר (כגון 'משקי בית משגשגים', 'מתוחים כלכלית'). "
        "המטרה היא להנגיש את הממצאים גם לקהל לא-טכני.\n\n"

        "✍️ חשוב צעד אחר צעד וענה בעברית ברורה, עם הסבר אינטואיטיבי ומובנה שמדגיש קשרים, חריגים או מגמות בדאטה בלבד."
    )

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.3,
    )

    return response.choices[0].message.content.strip()

In [3]:
def build_context_text() -> str:
    parts = []

    # 📊 Cluster Summary
    parts.append("📊 סיכום קלאסטרים (מאפיינים ממוצעים לכל קבוצה):\n")
    for row in cluster_summary:
        parts.append(
            f"Cluster {row['prediction']}:\n"
            f"  • שכר ממוצע: {row['avg_salary']:.0f} ש\"ח\n"
            f"  • חיסכון ממוצע: {row['avg_savings']:.0f} ש\"ח\n"
            f"  • דירוג כלכלי ממוצע: {row['avg_economic_class']:.2f}\n"
            f"  • יחס הוצאה/הכנסה: {row['avg_expense_ratio']:.2f}\n"
            f"  • שנות השכלה ממוצעות: {row['avg_education_years_filled']:.2f}\n"
            f"  • סקטור תעסוקתי ממוצע (אינדקס): {row['avg_work_sector_index']:.2f}\n"
        )

    # 📦 Cluster Sizes
    parts.append("\n📦 גודל כל קלאסטר (מספר משקי בית):\n")
    for row in cluster_sizes:
        parts.append(f"  • Cluster {row['prediction']}: {row['count']} משקי בית")

    # 💰 Salary Deciles and Dominant Work Sectors
    parts.append("\n\n💰 עשירוני שכר וסקטורים מובילים:\n")
    for row in salary_decile_sector_summary:
        parts.append(
            f"Decile {row['decile']}:\n"
            f"  • שכר ממוצע: {row['avg_salary']:.0f} ש\"ח\n"
            f"  • שנות השכלה ממוצעות: {row['avg_education_years']:.2f}\n"
            f"  • סקטור מוביל: {row['top_sector']} ({row['percent_top_sector']:.1f}%)\n"
            f"  • סקטור שני: {row['second_sector']} ({row['percent_second_sector']:.1f}%)\n"
        )

    # 🏭 Work Sector Mapping
    parts.append("\n🏭 מיפוי סקטורים (המרת קודים לסקטורים):\n")
    parts.append(
        "  • C → 0 : שירותים / מעורב\n"
        "  • B → 1 : הייטק / מקצועי\n"
        "  • A → 2 : מנהלים / צווארון לבן\n"
        "  • E → 3 : בכירים / פיננסים\n"
        "  • D → 4 : עבודה פיזית / הכנסה נמוכה"
    )

    return "\n".join(parts)

In [4]:
import json
from pathlib import Path

BASE_DIR = Path(".")
OUTPUT_JSON_DIR = BASE_DIR / "output" / "json"

def load_json(name: str):
    path = OUTPUT_JSON_DIR / name
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_context_text() -> str:
    """
    בונה טקסט קונטקסט קריא ל-LLM,
    כולל הסבר קצר לפני כל 'טבלה' שמגיעה מה-JSON.
    """
    # 1) סיכום קלאסטרים
    clusters = load_json("cluster_summary.json")
    clusters_text_lines = [
        "1) סיכום קלאסטרים (K-Means על משקי בית):",
        "   לכל קלאסטר יש ערכים ממוצעים של שכר, חיסכון, הוצאות, השכלה וסקטור עבודה.",
        "   השדות העיקריים:",
        "   - prediction: מספר הקלאסטר (0–3).",
        "   - avg_salary: שכר חודשי ממוצע.",
        "   - avg_savings: סכום חיסכון ממוצע.",
        "   - avg_expense_ratio: יחס הוצאה/שכר (גבוה=הכנסה 'נחנקת').",
        "   - avg_education_years_filled: שנות לימוד ממוצעות.",
        "   - avg_work_sector_index: קידוד סקטור (0–4).",
        "   - avg_economic_class: מדד מעמד כלכלי (נמוך–גבוה).",
        "",
        "   קלאסטרים (תיאור מספרי):"
    ]
    for row in clusters:
        clusters_text_lines.append(
            f"   • Cluster {row['prediction']}: "
            f"salary~{row['avg_salary']:.0f}, "
            f"savings~{row['avg_savings']:.0f}, "
            f"econ_class~{row['avg_economic_class']:.1f}, "
            f"edu_years~{row['avg_education_years_filled']:.1f}, "
            f"expense_ratio~{row['avg_expense_ratio']:.2f}"
        )
    clusters_text = "\n".join(clusters_text_lines)

    # 2) גודל כל קלאסטר
    cluster_sizes = load_json("cluster_sizes.json")
    sizes_text_lines = [
        "",
        "2) גודל כל קלאסטר (מספר משקי בית בכל קלאסטר):"
    ]
    for row in cluster_sizes:
        sizes_text_lines.append(f"   • Cluster {row['prediction']}: {row['count']} משקי בית")
    sizes_text = "\n".join(sizes_text_lines)

    # 3) התפלגות סקטורי עבודה לפי עשירוני שכר (אם יש לך JSON כזה)
    try:
        deciles = load_json("decile_sector_summary.json")
        deciles_text_lines = [
            "",
            "3) התפלגות סקטורי עבודה לפי עשירוני שכר:",
            "   לכל שורה: עשירון (0–10%, 10–20% וכו'), שכר ממוצע, שנות לימוד ממוצעות, וסקטורים דומיננטיים.",
        ]
        for row in deciles:
            deciles_text_lines.append(
                f"   • Decile {row['decile']}: "
                f"avg_salary={row['avg_salary']:.0f}, "
                f"avg_education_years={row['avg_education_years']:.1f}, "
                f"top_sector={row['top_sector']} ({row['percent_top_sector']:.1f}%), "
                f"second_sector={row['second_sector']} ({row['percent_second_sector']:.1f}%)"
            )
        deciles_text = "\n".join(deciles_text_lines)
    except FileNotFoundError:
        deciles_text = "\n(אין כרגע קובץ decile_sector_summary.json, מדלגים על חלק זה.)"

    # 4) מילון סקטורים
    sector_map = {
        0: "C – שירותים ומקצועות מעורבים",
        1: "B – מקצועות מקצועיים / הייטק בינוני",
        2: "A – מקצועות לבנים / מנהלים",
        3: "E – הייטק/הכנסה גבוהה מאוד",
        4: "D – עבודות ידניות / כחול-לבן / שכר נמוך"
    }
    sectors_text_lines = [
        "",
        "4) מיפוי סקטורי עבודה לפי אינדקס:",
    ]
    for idx, desc in sector_map.items():
        sectors_text_lines.append(f"   • {idx}: {desc}")
    sectors_text = "\n".join(sectors_text_lines)

    # לחבר הכל
    full_context = "\n".join([clusters_text, sizes_text, deciles_text, sectors_text])
    return full_context

In [5]:
answer = ask_econ_assistant(
    "תסבירי בקצרה את ארבעת הקלאסטרים: מי הם, מה רמת ההכנסה שלהם, "
    "ומה ההבדלים העיקריים ביניהם?"
)
print(answer)

בהסתמך על נתוני הקלאסטרים, ניתן לתאר את ארבעת קבוצות משקי הבית באופן הבא:

1. **קלאסטר 0 – "משקי בית במתח כלכלי גבוה"**  
   - שכר ממוצע: כ-2,489 ש"ח בלבד, הנמוך ביותר מבין הקבוצות.  
   - חיסכון ממוצע: כ-4,934 ש"ח, נמוך מאוד.  
   - יחס הוצאה/הכנסה: 0.97 – כמעט כל ההכנסה מוצאת על הוצאות, מצב כלכלי לחוץ מאוד.  
   - שנות השכלה: 12.5 בממוצע, הנמוך ביותר.  
   - סקטור עבודה: שירותים ומקצועות מעורבים (קוד 0).  
   - מעמד כלכלי: 1.3 – הנמוך ביותר.  
   - גודל: 1,143 משקי בית.  
   **מסקנה:** זוהי קבוצה של משקי בית עם הכנסה נמוכה מאוד, חיסכון זעום ונטייה להוצאות כמעט שוות להכנסה, מה שמצביע על קושי כלכלי משמעותי. רמת ההשכלה נמוכה יחסית, והעבודה בעיקר בשירותים ומקצועות מעורבים.

2. **קלאסטר 1 – "משקי בית משגשגים מאוד"**  
   - שכר ממוצע: כ-8,074 ש"ח, הגבוה ביותר.  
   - חיסכון ממוצע: כ-53,876 ש"ח, גבוה מאוד.  
   - יחס הוצאה/הכנסה: 0.43 – הוצאות נמוכות יחסית להכנסה, מצב כלכלי נוח.  
   - שנות השכלה: 18.4 בממוצע, הגבוה ביותר.  
   - סקטור עבודה: הייטק/הכנסה גבוהה מאוד (קוד 3).  
   - מעמד כלכל

In [6]:
q1 = "האם יש קלאסטר שבו רמת ההשכלה גבוהה יחסית, אבל השכר והחיסכון נמוכים? מה זה אומר?"
q2 = "באיזה קבוצה את מזהה הכי הרבה סיכון להידרדרות כלכלית על סמך יחס הוצאה להכנסה וחיסכון נמוך?"
q3 = "איך היית מתארת את פערי המעמד הכלכלי בין הקלאסטרים במילים, לא רק במספרים?"
questions = [q1, q2, q3]

for i, q in enumerate(questions, start=1):
    print(f"\n==================== שאלה {i} ====================\n")
    print("❓", q, "\n")
    answer = ask_econ_assistant(q)
    print("🧠 תשובה:\n")
    print(answer)
    print("\n" + "="*60 + "\n")
print(ask_econ_assistant(q1))
print(ask_econ_assistant(q2))
print(ask_econ_assistant(q3))


==================== שאלה 1 ====================

❓ האם יש קלאסטר שבו רמת ההשכלה גבוהה יחסית, אבל השכר והחיסכון נמוכים? מה זה אומר? 

🧠 תשובה:

בהתבסס על הנתונים שסופקו, ננתח את הקלאסטרים ונבדוק האם קיים קלאסטר שבו רמת ההשכלה גבוהה יחסית, אך השכר והחיסכון נמוכים.

---

### שלב 1: מתן שמות אינטואיטיביים לקלאסטרים לפי מאפיינים מרכזיים

- **Cluster 0 (1143 משקי בית):**  
  שכר נמוך מאוד (כ-2,489 ש"ח), חיסכון נמוך (4,934 ש"ח), יחס הוצאה/שכר גבוה מאוד (0.97), שנות השכלה בינוניות (12.5), מעמד כלכלי נמוך (1.3).  
  → ניתן לכנות: **"משקי בית במתח כלכלי חמור"**

- **Cluster 1 (983 משקי בית):**  
  שכר גבוה מאוד (8,074 ש"ח), חיסכון גבוה מאוד (53,876 ש"ח), יחס הוצאה/שכר נמוך (0.43), שנות השכלה גבוהות מאוד (18.4), מעמד כלכלי גבוה (6.9).  
  → ניתן לכנות: **"משקי בית משגשגים"**

- **Cluster 2 (2506 משקי בית):**  
  שכר גבוה יחסית (6,154 ש"ח), חיסכון גבוה (22,633 ש"ח), יחס הוצאה/שכר נמוך יחסית (0.48), שנות השכלה גבוהות (17.1), מעמד כלכלי בינוני-גבוה (4.9).  
  → ניתן לכנות: **"משקי בית יציבים עם הש

In [7]:
q4 = "תן לכל קלאסטר שם שיווקי (למשל 'משקי בית מתוחים', 'משקי בית יציבים', 'משקי בית משגשגים') ותסביר למה בחרת בכל שם."
q5 = "הצע אסטרטגיה פיננסית שונה לכל קלאסטר: מה להציע להם כמוצר פיננסי או שירות ייעוץ?"
q6 = "אם אני בנק או חברת פינטק, איך היית ממליצה לי להשתמש בחלוקת הקלאסטרים האלה כדי לעצב קמפיין שיווקי חכם?"
questions = [q4, q5, q6]

for i, q in enumerate(questions, start=1):
    print(f"\n==================== שאלה {i} ====================\n")
    print("❓", q, "\n")
    answer = ask_econ_assistant(q)
    print("🧠 תשובה:\n")
    print(answer)
    print("\n" + "="*60 + "\n")
print(ask_econ_assistant(q4))
print(ask_econ_assistant(q5))
print(ask_econ_assistant(q6))


==================== שאלה 1 ====================

❓ תן לכל קלאסטר שם שיווקי (למשל 'משקי בית מתוחים', 'משקי בית יציבים', 'משקי בית משגשגים') ותסביר למה בחרת בכל שם. 

🧠 תשובה:

בהתבסס על הנתונים שסופקו, נבחן כל קלאסטר ונציע שם שיווקי מתאים עם הסבר כלכלי ברור:

---

### קלאסטר 0:  
- שכר ממוצע נמוך יחסית (כ-2,489 ש"ח)  
- חיסכון נמוך (4,934 ש"ח)  
- יחס הוצאה/הכנסה גבוה מאוד (0.97) – כמעט כל ההכנסה מתבזבזת, חוסך מעט מאוד או בכלל לא  
- שנות השכלה נמוכות יחסית (12.5)  
- מעמד כלכלי נמוך (1.3)  
- סקטור עבודה: אינדקס 0 – שירותים ומקצועות מעורבים

**שם מוצע:**  
**"משקי בית מתוחים כלכלית"**

**הסבר:**  
משקי הבית בקבוצה זו מתאפיינים בשכר נמוך מאוד וביחס הוצאה כמעט שווה להכנסה, כלומר הם כמעט לא מצליחים לחסוך. רמת ההשכלה נמוכה יחסית, והמעמד הכלכלי נמוך מאוד. הם ככל הנראה מתמודדים עם לחצים כלכליים גבוהים, מתקשים לעמוד בהוצאות השוטפות. השם "מתוחים כלכלית" מדגיש את המתח הכלכלי והחוסר ברווחה כלכלית.

---

### קלאסטר 1:  
- שכר גבוה מאוד (8,074 ש"ח)  
- חיסכון גבוה מאוד (53,876 ש"ח)  
- יחס הוצאה